# Jan's Matchday · The training ground, as code

Every drill on the [Learn pages](https://jhvl88.github.io/matchday/learn/) in one notebook: the same formulas, the same data, the same numbers. Run it top to bottom (Runtime › Run all). It needs no GPU: everything here is counting and simulating, and the whole notebook takes a minute or two on a free CPU runtime.

**Part one:** 1 What 32% means · 2 From goals to a forecast · 3 What a chance is worth · 4 Three models, one average · 5 From odds to chances · 6 Grading · 7 The long run
**Part two:** 8 Is the table lying? · 9 Elo · 10 Calibration · 11 How much to bet · Extra time: penalties

The real data comes from the repository's own branches: football-data.co.uk's season files (`source-data`) and the published models' out-of-sample predictions (`backtest-results`). Section 0 fetches both and rebuilds `data/learn.json` with the same script the site uses.

## 0 · Setup

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Where things live. In Colab the defaults are fine; the environment variables let the notebook run elsewhere.
REPO = Path(os.environ.get("MATCHDAY_REPO", "/content/matchday"))
SOURCE = Path(os.environ.get("MATCHDAY_SOURCE", "/content/source"))      # football-data.co.uk season files
RESULTS = Path(os.environ.get("MATCHDAY_RESULTS", "/content/results"))   # published models' backtest predictions
WORK = Path(os.environ.get("MATCHDAY_WORK", "/content/work"))
WORK.mkdir(parents=True, exist_ok=True)

def run(*cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, cwd=cwd)

if not REPO.exists():
    run("git", "clone", "--depth", "1", "https://github.com/JHvL88/matchday", REPO)
if not SOURCE.exists() or not RESULTS.exists():
    run("git", "fetch", "--depth", "1", "origin", "source-data:source-data", "backtest-results:backtest-results", cwd=REPO)
    if not SOURCE.exists():
        run("git", "worktree", "add", SOURCE, "source-data", cwd=REPO)
    if not RESULTS.exists():
        run("git", "worktree", "add", RESULTS, "backtest-results", cwd=REPO)

sys.path.insert(0, str(REPO / "analysis" / "pipeline"))

# The site's colours: home win blue, draw grey, away win orange; ink for my models, grey for the bookmakers.
HOME, DRAW, AWAY, INK, MUTED = "#4f8fee", "#868d9c", "#f5a155", "#1b1e25", "#8a8f99"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 10})
rng = np.random.default_rng(2026)
print("repo:", REPO, "| source:", SOURCE, "| results:", RESULTS)

In [ ]:
# The match table and data/learn.json, rebuilt from the sources with the site's own scripts.
MATCHES = WORK / "matches.csv"
LEARN = WORK / "learn.json"
run(sys.executable, REPO / "analysis/pipeline/build_matches.py", SOURCE, MATCHES)
run(sys.executable, REPO / "analysis/pipeline/learn_data.py", MATCHES, RESULTS / "xg", LEARN, "--as-of", pd.Timestamp.today().date())
learn = json.loads(LEARN.read_text())
matches = pd.read_csv(MATCHES, parse_dates=["date"])
print(len(matches), "matches in the table;", len(learn["matches"]), "graded backtest forecasts")

## 1 · What 32% means

My forecast for Hamburg against Stuttgart (matchday 6, 2026/27): home win 32%, draw 24%, away win 44%. Play it once and you learn nothing. Play it many times and the counts settle on the forecast.

In [ ]:
forecast = np.array([0.32, 0.24, 0.44])
labels = ["Hamburg win", "Draw", "Stuttgart win"]

fig, axes = plt.subplots(1, 3, figsize=(11, 2.8), sharey=True)
for ax, n in zip(axes, [1, 100, 10_000]):
    share = np.bincount(rng.choice(3, size=n, p=forecast), minlength=3) / n
    ax.bar(labels, share, color=[HOME, DRAW, AWAY])
    ax.scatter(labels, forecast, marker="_", s=900, color=INK, zorder=3, label="forecast")
    ax.set_title(f"{n:,} replay{'s' if n > 1 else ''}")
    ax.tick_params(axis="x", labelsize=8)
axes[0].set_ylabel("share of replays")
axes[0].legend(frameon=False)
plt.tight_layout()

## 2 · From goals to a forecast

Two averages, one per side, spread over whole numbers with the Poisson distribution. Every cell of the grid is a score; add up the cells below the diagonal and you have the home win.

In [ ]:
from scipy.stats import poisson

def score_grid(home_goals, away_goals, max_goals=14):
    g = np.arange(max_goals + 1)
    return np.outer(poisson.pmf(g, home_goals), poisson.pmf(g, away_goals))

def outcome(grid):
    return np.array([np.tril(grid, -1).sum(), np.trace(grid), np.triu(grid, 1).sum()])

grid = score_grid(1.5, 1.2)
print("home, draw, away:", outcome(grid).round(3))            # 0.44, 0.25, 0.30
top = np.unravel_index(grid.argmax(), grid.shape)
print("likeliest score:", top, f"{grid.max():.1%}")         # 1-1 at about 12%
print("over 2.5 goals:", round(sum(grid[i, j] for i in range(15) for j in range(15) if i + j >= 3), 3))

plt.figure(figsize=(4, 3.4))
plt.imshow(grid[:6, :6] * 100, cmap="Blues")
for i in range(6):
    for j in range(6):
        plt.text(j, i, f"{grid[i, j] * 100:.0f}", ha="center", va="center", fontsize=8)
plt.xlabel("away goals"); plt.ylabel("home goals"); plt.title("Chance of each score, %"); plt.grid(False)
plt.colorbar(fraction=.046)

In [ ]:
# The site's models add Dixon–Coles' small correction for low scores (bl_models.score_matrix).
from bl_models import outcome_probs, score_matrix
print("plain Poisson:", outcome(grid).round(3))
print("Dixon–Coles, rho = -0.13:", outcome_probs(score_matrix(1.5, 1.2, -0.13)).round(3))

## 3 · What a chance is worth (a toy xG model)

The shot map on the Learn page prices a shot from two things only: how much of the goal you can see (the angle) and how far away you are. Real xG models know much more; this one is tuned by hand to land near the usual figures.

In [ ]:
POST = 3.66  # half the goal, metres

def toy_xg(x, d):
    """x: metres left/right of the middle of the goal, d: metres out from the goal line."""
    distance = np.hypot(x, d)
    angle = np.arctan2(2 * POST * d, x**2 + d**2 - POST**2)
    angle = np.where(angle < 0, angle + np.pi, angle)
    return 1 / (1 + np.exp(-(-2.45 + 1.86 * angle - 0.07 * distance)))

for name, x, d in [("six-yard box", 0, 5.5), ("penalty spot", 0, 11), ("edge of the box", 0, 17),
                   ("tight angle", 9, 2.5), ("25 metres out", -4, 25)]:
    print(f"{name:16s} xG {float(toy_xg(x, d)):.2f}")

X, D = np.meshgrid(np.linspace(-30, 30, 241), np.linspace(0.3, 33, 131))
plt.figure(figsize=(6, 3.4))
plt.contourf(X, D, toy_xg(X, D), levels=[0, .02, .05, .1, .2, .3, .5, 1], cmap="Blues")
plt.colorbar(label="xG")
plt.gca().invert_yaxis(); plt.gca().set_aspect("equal")
plt.plot([-20.16, -20.16, 20.16, 20.16], [0, 16.5, 16.5, 0], color=INK, lw=1)
plt.plot([-9.16, -9.16, 9.16, 9.16], [0, 5.5, 5.5, 0], color=INK, lw=1)
plt.title("Toy xG over the attacking third (goal at the top)"); plt.grid(False)

## 4 · Three models, one average

The latest matchday's published forecasts (`data/forecasts.json`): the three models and their plain average, next to the bookmakers.

In [ ]:
fc = json.loads((REPO / "data/forecasts.json").read_text())
md_ = max(fc["matchdays"], key=int)
rows = []
for match_id, m in fc["matchdays"][md_]["matches"].items():
    inv = 1 / np.array(m["market"]["odds"])
    rows.append(dict(match=f'{m["home"]} – {m["away"]}', ensemble=m["model"][0], market=(inv / inv.sum())[0],
                     **{k: v[0] for k, v in m["models"].items()}))
table = pd.DataFrame(rows).set_index("match").sort_values("ensemble")
ax = table[["dixonColes", "bayesian", "xgboost"]].plot(style=["o", "s", "^"], color=MUTED, figsize=(7, 4), rot=0, legend=False)
table["ensemble"].plot(ax=ax, style="D", color=INK, ms=7)
table["market"].plot(ax=ax, style="o", mfc="none", color=INK, ms=9)
ax.set_xticks(range(len(table))); ax.set_xticklabels(table.index, rotation=60, ha="right", fontsize=8)
ax.set_ylabel("home win"); ax.legend(["Dixon–Coles", "Shots to goals", "XGBoost", "Ensemble", "Bookmakers"], frameon=False, fontsize=8)
ax.set_title(f"Matchday {md_}: home-win chance by model")
table.round(3)

## 5 · From odds to chances

1 ÷ odds for each outcome, then scale down until they add up to 100%. What was left over is the bookmaker's margin.

In [ ]:
def market_probs(odds):
    inverse = 1 / np.asarray(odds, float)
    return inverse / inverse.sum(), inverse.sum() - 1     # chances, margin

probs, margin = market_probs([3.80, 4.20, 1.80])          # Hamburg – Stuttgart at Bet365
print("market:", probs.round(3), f"margin {margin:.1%}")
print("fair odds:", (1 / probs).round(2))
print("coin at 1.90 / 1.90:", market_probs([1.90, 1.90]))   # 50/50 and a 5.3% margin

## 6 · Grading: the ranked probability score

Adds the forecast up step by step (home, then home plus draw), does the same for what happened, squares the gaps, averages. 0 is perfect; a shrug scores about 0.24.

In [ ]:
from bl_models import rps

home_win = 0
print("A (60% on the draw):", rps(np.array([.2, .6, .2]), [home_win])[0])   # 0.34
print("B (60% away win):   ", rps(np.array([.2, .2, .6]), [home_win])[0])   # 0.50
print("a shrug, each result:", rps(np.full((3, 3), 1 / 3), [0, 1, 2]).round(3))

## 7 · The long run

€1 bets at odds of 1.90. The share of bettors in profit after *n* bets is exact (binomial); the lines are simulated.

In [ ]:
from scipy.stats import binom

def share_in_profit(n, p, odds=1.90):
    need = np.floor(n / odds) + 1           # more wins than n / odds
    return binom.sf(need - 1, n, p)

for p in [0.50, 0.526, 0.54]:
    print(f"right {p:.1%}: ", {n: round(float(share_in_profit(n, p)), 3) for n in [100, 500, 2000]})
print("a coin-flipper at least €8 up after 100 bets:", round(float(binom.sf(np.ceil(108 / 1.9) - 1, 100, .5)), 3))

wins = rng.random((40, 500)) < 0.50
paths = np.cumsum(np.where(wins, 0.90, -1.0), axis=1)
plt.figure(figsize=(7, 3))
plt.plot(paths.T, color=MUTED, lw=.8, alpha=.6)
plt.plot(np.arange(500) * (0.5 * 1.9 - 1), color=INK, lw=2, ls="--", label="average")
plt.axhline(0, color=INK, lw=.8); plt.xlabel("bets"); plt.ylabel("€"); plt.legend(frameon=False)
plt.title("40 coin-flippers at odds of 1.90")

## 8 · Is the table lying?

How spread out the real table is, matchday by matchday (every complete season in the match table), against eighteen identical clubs who play with dice. The dice use the real home/draw/away shares.

In [ ]:
H, D_, A = learn["table"]["homeDrawAway"]
real = np.array(learn["table"]["sd"])

def clone_season(rng, clubs=18):
    pts = np.zeros((35, clubs))
    order, rounds = list(range(clubs)), []
    for r in range(clubs - 1):                               # circle method, then the return legs
        pairs = [(order[i], order[clubs - 1 - i]) for i in range(clubs // 2)]
        rounds.append([(b, a) for a, b in pairs] if r % 2 else pairs)
        order = [order[0], order[-1]] + order[1:-1]
    rounds += [[(b, a) for a, b in rd] for rd in rounds]
    for md, rd in enumerate(rounds, 1):
        pts[md] = pts[md - 1]
        for h, a in rd:
            u = rng.random()
            if u < H: pts[md, h] += 3
            elif u < H + D_: pts[md, h] += 1; pts[md, a] += 1
            else: pts[md, a] += 3
    return pts

sims = np.array([clone_season(rng) for _ in range(1000)])
clones = sims[:, 1:, :].std(axis=2).mean(axis=0)
final = np.sort(sims[:, 34, :], axis=1)
print(f"clones: champion {final[:, -1].mean():.1f} pts, bottom {final[:, 0].mean():.1f}")
print(f"real:   champion {learn['table']['champion']} pts, bottom {learn['table']['bottom']}")
print("clones as spread out as the real league after 5, 17, 34 matchdays:", (clones / real)[[4, 16, 33]].round(2))

plt.figure(figsize=(7, 3.2))
plt.plot(range(1, 35), real, color=INK, lw=2, label="Real Bundesliga")
plt.plot(range(1, 35), clones, color=MUTED, lw=2, ls="--", label="League of clones")
plt.xlabel("matchday"); plt.ylabel("spread of points (sd)"); plt.legend(frameon=False)

## 9 · Elo

The update the models use (`features.elo_ratings`): expected share of the points from the rating gap plus 60 for home, times 20, more for wins by two or more. The ladder is the real one, rebuilt in section 0.

In [ ]:
def elo_update(home, away, home_goals, away_goals, k=20, home_adv=60):
    expected = 1 / (1 + 10 ** (-(home + home_adv - away) / 400))
    gd = home_goals - away_goals
    result = 1.0 if gd > 0 else 0.5 if gd == 0 else 0.0
    mult = 1.0 if abs(gd) <= 1 else 1.5 if abs(gd) == 2 else (11 + abs(gd)) / 8
    delta = k * mult * (result - expected)
    return home + delta, away - delta

print("1700 v 1350, draw:", np.round(elo_update(1700, 1350, 1, 1), 1))     # the favourite pays about 8 points
print("1350 v 1700, 3-0: ", np.round(elo_update(1350, 1700, 3, 0), 1))     # a shock moves about 30

ladder = pd.DataFrame(learn["elo"]["clubs"], columns=["club", "now", "a year ago"]).set_index("club")
ax = ladder.iloc[::-1].plot(kind="barh", y="now", color=MUTED, figsize=(6, 5), legend=False, xlim=(1300, 1750))
ax.scatter(ladder.iloc[::-1]["a year ago"], range(len(ladder)), color=INK, zorder=3, s=14, label="a year ago")
ax.set_title(f"Elo after {learn['elo']['asOf']}"); ax.legend(frameon=False)

## 10 · Calibration and the boldness dial

Every out-of-sample forecast the published ensemble has made since 2019/20, loaded the way `evaluate.py` grades it. The dial raises every probability to a power *b* and renormalises: *b* > 1 is bolder, *b* < 1 shyer.

In [ ]:
from evaluate import cols, load
o = load(MATCHES, RESULTS / "xg")
o = o[o.hg.notna() & o[cols("avgc")].notna().all(axis=1)].reset_index(drop=True)
p_models, p_market, y = o[cols("ens")].to_numpy(), o[cols("avgc")].to_numpy(), o.y.to_numpy()
print(len(o), "matches; RPS models", rps(p_models, y).mean().round(4), "| bookmakers", rps(p_market, y).mean().round(4))

def bolder(p, b):
    q = np.clip(p, 1e-9, 1) ** b
    return q / q.sum(axis=1, keepdims=True)

def calibration(p, y, bins=10):
    hit, prob = np.eye(3)[y].ravel(), p.ravel()
    k = np.minimum((prob * bins).astype(int), bins - 1)
    return np.array([(prob[k == i].mean(), hit[k == i].mean(), (k == i).sum()) for i in range(bins) if (k == i).sum() >= 20])

dials = np.round(np.arange(0.5, 2.01, 0.01), 2)
scores = [rps(bolder(p_models, b), y).mean() for b in dials]
print("best dial:", dials[int(np.argmin(scores))], "RPS", round(min(scores), 4))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.plot([0, 1], [0, 1], ls="--", color=MUTED, lw=1)
for b, style in [(1.0, dict(color=INK)), (1.5, dict(color=AWAY)), (0.6, dict(color=HOME))]:
    c = calibration(bolder(p_models, b), y)
    a1.scatter(c[:, 0], c[:, 1], s=8 + c[:, 2] / 8, label=f"models ×{b}", **style)
c = calibration(p_market, y)
a1.scatter(c[:, 0], c[:, 1], s=8 + c[:, 2] / 8, facecolors="none", edgecolors=MUTED, label="bookmakers")
a1.set_xlabel("forecast"); a1.set_ylabel("how often it happened"); a1.legend(frameon=False, fontsize=8)
a2.plot(dials, scores, color=INK); a2.axhline(rps(p_market, y).mean(), color=MUTED, ls="--", label="bookmakers")
a2.set_xlabel("boldness b"); a2.set_ylabel("RPS (lower is better)"); a2.legend(frameon=False)
plt.tight_layout()

## 11 · How much to bet

Kelly: stake edge ÷ (odds − 1) of the pot. The world here: odds of 2.10, a true 50% chance (a 5% edge). What happens to the typical pot after 500 bets when your belief about your chance is off?

In [ ]:
def kelly(p, odds):
    return max(0.0, (p * odds - 1) / (odds - 1))

def final_pots(belief, fraction, true_p=0.50, odds=2.10, bets=500, bettors=4000, rng=rng):
    f = fraction * kelly(belief, odds)
    wins = (rng.random((bettors, bets)) < true_p).sum(axis=1)
    return np.maximum(0.01, 100 * (1 + f * (odds - 1)) ** wins * (1 - f) ** (bets - wins))   # a cent counts as gone

print("Kelly stake at a true 50%:", round(kelly(0.50, 2.10), 4))
beliefs = np.arange(0.48, 0.601, 0.005)
plt.figure(figsize=(7, 3.4))
for fraction, style in [(1.0, dict(color=INK, label="Kelly")), (0.5, dict(color=INK, ls="--", label="half Kelly")),
                        (2.0, dict(color=MUTED, label="double Kelly"))]:
    plt.plot(beliefs, [np.median(final_pots(b, fraction)) for b in beliefs], **style)
plt.axhline(100, color=MUTED, lw=.8); plt.axvline(0.5, color=MUTED, lw=.8, ls=":")
plt.yscale("log"); plt.ylim(0.01, 1000); plt.xlabel("what you think your chance is (truth: 50%)"); plt.ylabel("typical pot after 500 bets, €")
plt.legend(frameon=False)
for b in [0.50, 0.55]:
    print(f"belief {b:.0%}: Kelly median €{np.median(final_pots(b, 1)):.0f}, half Kelly €{np.median(final_pots(b, .5)):.0f}")

## Extra time · Penalties

Ten real matches from the backtest, drawn at random. Put your own forecasts in `mine` and see who would have scored.

In [ ]:
pick = rng.choice(len(o), 10, replace=False)
mine = np.tile([0.40, 0.28, 0.32], (10, 1))     # <- your forecasts here: home, draw, away for each match
shoot = pd.DataFrame({
    "match": (o.home + " – " + o.away).iloc[pick].values,
    "date": o.date.dt.date.iloc[pick].values,
    "score": (o.hg.astype(int).astype(str) + "-" + o.ag.astype(int).astype(str)).iloc[pick].values,
    "you": rps(mine, y[pick]), "models": rps(p_models[pick], y[pick]), "bookmakers": rps(p_market[pick], y[pick]),
})
shoot["closest"] = shoot[["you", "models", "bookmakers"]].idxmin(axis=1)
print(shoot[["you", "models", "bookmakers"]].mean().round(3).to_dict())
shoot.round(3)